# testing the full system

One game, one reply at a time. **Load** reads a directory under `weights/full/` (`s1.pt`, the readout, and `gemma/`) through `S1S2.load`. A 3-point checkpoint fails here and the log names `python -m neural_net.convert_to_sv`.

**New game** deals `new_multi_gold_game`. Gold defaults to random (0–3) and the opening to any, so a sealed empty room, a room with no gold, and a room with several golds all show up. The question box is filled with the default line: the move beginning on the first turn, then "Move to your target." after that. The user text includes the board report when the previous reply ended in `[MOVE]`. S1 stays dormant unless this reply ends in `[MOVE]`. Then S1 runs for 10 seconds, or 20 no-ops in a row, or until the agent exits. Walking out ends the game. A reply that ends in `[END_GAME]` ends the game without waking S1; that is the win on a sealed empty board and a loss otherwise. The board shows the end state.

**Agent turn** sends the question box once. **Auto-play** sends the default line, up to 12 turns, and stops when the game is over. The presets are the beginning lines, the two "your target" lines, and the direction and hour questions. Choosing one fills the box. It does not run the model.

Tokens of each turn are clickable. A click draws that token's **s** in blue and **v** in red on the frame S2 saw, not on the board after the move. Click the same token again to return to the end state. The token that completed `[MOVE]` is outlined in red. The token that completed `[END_GAME]` is outlined in green.

**Replay** rebuilds the latest move from its recorded actions and plays those frames, then returns to the end state. **End game** drops the session. The notepad is the in-memory `[REMEMBER]` notes. Control comes back after every turn.

Run this on the GPU box. The cell needs the repo environment (`pygame`, `torch`, `ipywidgets`).


In [ ]:
from __future__ import annotations

import html
import io
import os
import sys
import time
import traceback

os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import ipywidgets as widgets
import numpy as np
from IPython.display import HTML, display
from PIL import Image, ImageDraw

from agent.modes import (
    S2_BEGINNING_MOVE_USER,
    S2_BEGINNING_USER,
    S2_LOOK_TARGET_USER,
    S2_MOVE_TARGET_USER,
)
from neural_net.loop import S1S2
from neural_net.paths import weights_root
from neural_net.render import canonical_frame
from neural_net.s2_coord_train import QUESTION_DIRECTION, QUESTION_HOUR
from neural_net.s2_session import S2GameSession

BOARD_PX = 560
_COLOR = {"s": (30, 90, 255), "v": (255, 0, 0)}

state = {
    "net": None,
    "session": None,
    "turns": [],
    "end_frame": None,
    "selected": None,
    "busy": False,
    "ignore_through": 0,
    "gen": 0,
}


def _full_dirs():
    root = weights_root() / "full"
    if not root.is_dir():
        return []
    found = []
    for path in root.iterdir():
        if not path.is_dir():
            continue
        if (
            (path / "s1.pt").is_file()
            and (path / "localizer.pt").is_file()
            and (path / "coord_embed.pt").is_file()
            and (path / "gemma").is_dir()
        ):
            found.append(path)
    found.sort(key=lambda item: item.stat().st_mtime, reverse=True)
    return found


def _options(paths):
    if not paths:
        return [("(no weights/full checkpoint)", "")]
    root = weights_root()
    options = []
    for path in paths:
        try:
            label = path.relative_to(root).as_posix()
        except ValueError:
            label = path.as_posix()
        options.append((label, str(path)))
    return options


def _say(text):
    log.append_stdout(text if text.endswith("\n") else text + "\n")


def _set_busy(flag):
    state["busy"] = flag
    for widget in controls:
        widget.disabled = flag


def _png(rgb):
    buf = io.BytesIO()
    Image.fromarray(np.ascontiguousarray(rgb), mode="RGB").save(buf, format="PNG")
    return buf.getvalue()


def _dot(draw, size, x, y, color):
    width, height = size
    px = float(x) * (width - 1)
    py = (1.0 - float(y)) * (height - 1)
    r = 8
    draw.ellipse((px - r, py - r, px + r, py + r), fill=color, outline=(0, 0, 0))


def _overlay(frame, token):
    image = Image.fromarray(np.ascontiguousarray(frame), mode="RGB")
    draw = ImageDraw.Draw(image)
    _piece, s_xy, v_xy, _eos, _move, _end = token
    if s_xy is not None:
        _dot(draw, image.size, s_xy[0], s_xy[1], _COLOR["s"])
    if v_xy is not None:
        _dot(draw, image.size, v_xy[0], v_xy[1], _COLOR["v"])
    return np.asarray(image)


def _show_board():
    selected = state["selected"]
    if selected is not None:
        turn_i, tok_i = selected
        turn = state["turns"][turn_i]
        board.value = _png(_overlay(turn.frame, turn.tokens[tok_i]))
        return
    frame = state["end_frame"]
    if frame is None:
        board.value = b""
        return
    board.value = _png(frame)


def _visible(piece):
    if piece == "":
        return "∅"
    return piece.replace(" ", "\u00a0")


def _token_markup():
    if not state["turns"]:
        return "<div class='fs-tokens'>Agent turn produces clickable tokens.</div>"
    gen = state["gen"]
    selected = state["selected"]
    blocks = []
    for turn_i, turn in enumerate(state["turns"]):
        parts = [f"<div class='fs-turn'>turn {turn_i + 1}</div>"]
        for tok_i, token in enumerate(turn.tokens):
            piece, _s, _v, eos, move_close, end_close = token
            cls = "fs-tok"
            if eos:
                cls += " fs-eos"
            if move_close:
                cls += " fs-move"
            if end_close:
                cls += " fs-end"
            if selected == (turn_i, tok_i):
                cls += " fs-on"
            parts.append(
                f"<span class='{cls}' data-gen='{gen}' "
                f"data-turn='{turn_i}' data-i='{tok_i}'>"
                f"{html.escape(_visible(piece))}</span>"
            )
        blocks.append("".join(parts))
    return "<div class='fs-tokens'>" + "<br>".join(blocks) + "</div>"


def _show_tokens():
    token_html.value = _token_markup()


def _show_log():
    blocks = []
    for turn_i, turn in enumerate(state["turns"]):
        blocks.append(
            f"<div class='fs-msg'><b>You, turn {turn_i + 1}</b>"
            f"<pre>{html.escape(turn.user_text)}</pre>"
            f"<b>Agent</b><pre>{html.escape(turn.text)}</pre></div>"
        )
    conv.value = "".join(blocks) if blocks else "<div class='fs-msg'>No turns yet.</div>"


def _show_notepad():
    session = state["session"]
    if session is None:
        notepad.value = "<pre>No game.</pre>"
        return
    notepad.value = f"<pre>{html.escape(session.notepad())}</pre>"


def _latest_move():
    for turn in reversed(state["turns"]):
        if turn.move is not None:
            return turn.move
    return None


def on_refresh(_):
    if state["busy"]:
        return
    paths = _full_dirs()
    ckpt.options = _options(paths)
    _say(f"{len(paths)} full checkpoint(s) under {weights_root() / 'full'}")


def on_load(_):
    if state["busy"]:
        return
    path = ckpt.value
    if not path:
        status.value = "No full checkpoint selected."
        return
    _set_busy(True)
    status.value = "Loading S1 and S2."
    try:
        state["net"] = S1S2.load(path)
        _say(f"loaded {path}")
        status.value = "New game, then Agent turn."
    except Exception:
        status.value = "Load failed."
        _say(traceback.format_exc())
        raise
    finally:
        _set_busy(False)


def on_new(_):
    if state["busy"]:
        return
    if state["net"] is None:
        status.value = "Load a full checkpoint first."
        return
    _set_busy(True)
    status.value = "Dealing a board."
    try:
        session = S2GameSession(
            state["net"],
            n_gold=gold.value,
            opening=str(opening.value),
        )
        state["session"] = session
        state["turns"] = []
        state["selected"] = None
        state["gen"] += 1
        state["end_frame"] = canonical_frame(session.game)
        question.value = session.next_question()
        _show_board()
        _show_tokens()
        _show_log()
        _show_notepad()
        _say(
            f"new game  gold={len(session.game.settings.gold)}  "
            f"opening={opening.value}"
        )
        status.value = "Agent turn sends the default question."
    except Exception:
        status.value = "New game failed."
        _say(traceback.format_exc())
        raise
    finally:
        _set_busy(False)


def on_preset(change):
    if change["name"] != "value":
        return
    question.value = _PRESETS[change["new"]]


def _over_line(outcome):
    verdict = "won" if outcome["won"] else "lost"
    return f"Game over: {outcome['reason']}, {verdict}."


def _after_turn(result):
    session = state["session"]
    state["turns"].append(result)
    state["gen"] += 1
    state["end_frame"] = canonical_frame(session.game)
    _show_log()
    _show_tokens()
    _show_notepad()
    _show_board()
    if result.outcome is not None:
        status.value = _over_line(result.outcome)
    else:
        move = result.move
        if move is None:
            status.value = (
                f"Reply finished. No [MOVE]. "
                f"{result.gold_remaining} gold(s) remain."
            )
        else:
            status.value = (
                f"{move.n_steps} steps, {move.elapsed:.2f}s, "
                f"stopped by {move.stopped_by}, "
                f"gold eaten {move.collected}, "
                f"{result.gold_remaining} remaining"
                + (", agent exited." if result.exited else ".")
            )
    if not session.over:
        question.value = session.next_question()
    _say(status.value)


def on_turn(_):
    if state["busy"]:
        return
    session = state["session"]
    if session is None:
        status.value = "Start a game first."
        return
    if session.over:
        status.value = _over_line(session.outcome)
        return
    text = question.value.strip()
    if not text:
        status.value = "The question is empty."
        return
    _set_busy(True)
    status.value = "S2 is writing."
    state["selected"] = None
    try:
        _after_turn(session.turn(text))
    except Exception:
        status.value = "Agent turn failed."
        _say(traceback.format_exc())
        raise
    finally:
        _set_busy(False)


def on_auto(_):
    if state["busy"]:
        return
    session = state["session"]
    if session is None:
        status.value = "Start a game first."
        return
    if session.over:
        status.value = _over_line(session.outcome)
        return
    _set_busy(True)
    status.value = "Auto-play."
    state["selected"] = None
    try:
        for result in session.play(max_turns=12):
            _after_turn(result)
        if not session.over:
            status.value = "Auto-play stopped at 12 turns. The game is still open."
            _say(status.value)
    except Exception:
        status.value = "Auto-play failed."
        _say(traceback.format_exc())
        raise
    finally:
        _set_busy(False)


def on_replay(_):
    if state["busy"]:
        return
    session = state["session"]
    move = _latest_move()
    if session is None or move is None:
        status.value = "No move to replay."
        return
    _set_busy(True)
    state["selected"] = None
    status.value = "Replaying the latest move."
    try:
        n = move.n_steps + 1
        stride = max(1, n // 80)
        last = None
        for index, frame in enumerate(session.replay(move)):
            last = frame
            if index % stride == 0 or index + 1 == n:
                board.value = _png(frame)
                time.sleep(0.03)
        if last is not None:
            state["end_frame"] = last
        _show_board()
        status.value = (
            f"Replay finished. {move.n_steps} steps, "
            f"stopped by {move.stopped_by}."
        )
    except Exception:
        status.value = "Replay failed."
        _say(traceback.format_exc())
        raise
    finally:
        _set_busy(False)


def on_end(_):
    if state["busy"]:
        return
    state["session"] = None
    state["turns"] = []
    state["end_frame"] = None
    state["selected"] = None
    state["gen"] += 1
    _show_board()
    _show_tokens()
    _show_log()
    _show_notepad()
    status.value = "Game ended."
    _say("game ended")


def on_pick(change):
    raw = change["new"]
    if not raw or raw.count(",") != 3:
        return
    seq_s, gen_s, turn_s, idx_s = raw.split(",")
    try:
        seq = int(seq_s)
        gen = int(gen_s)
        turn_i = int(turn_s)
        tok_i = int(idx_s)
    except ValueError:
        return
    if seq <= state["ignore_through"] or state["busy"]:
        return
    if gen != state["gen"]:
        return
    state["ignore_through"] = seq
    if turn_i < 0 or turn_i >= len(state["turns"]):
        return
    turn = state["turns"][turn_i]
    if tok_i < 0 or tok_i >= len(turn.tokens):
        return
    if state["selected"] == (turn_i, tok_i):
        state["selected"] = None
        readout.value = "Showing the board after the latest turn."
    else:
        state["selected"] = (turn_i, tok_i)
        piece, s_xy, v_xy, eos, move_close, end_close = turn.tokens[tok_i]
        readout.value = (
            f"turn {turn_i + 1} token {tok_i} &nbsp; {html.escape(piece)}<br>"
            f"s={s_xy} &nbsp; v={v_xy}"
            + ("<br>This token closed [MOVE]." if move_close else "")
            + ("<br>This token closed [END_GAME]." if end_close else "")
            + "<br>Drawn on the frame S2 saw."
        )
    _show_tokens()
    _show_board()


_PRESETS = {
    "begin_look": S2_BEGINNING_USER,
    "begin_move": S2_BEGINNING_MOVE_USER,
    "look_target": S2_LOOK_TARGET_USER,
    "move_target": S2_MOVE_TARGET_USER,
    "direction": QUESTION_DIRECTION,
    "hour": QUESTION_HOUR,
}

_paths = _full_dirs()
ckpt = widgets.Dropdown(
    options=_options(_paths),
    description="Checkpoint:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="520px"),
)
refresh_btn = widgets.Button(description="Refresh", icon="refresh")
load_btn = widgets.Button(description="Load", button_style="primary", icon="download")
gold = widgets.Dropdown(
    options=[("random", None)] + [(str(n), n) for n in (0, 1, 2, 3)],
    value=None,
    description="Gold:",
    style={"description_width": "50px"},
    layout=widgets.Layout(width="160px"),
)
opening = widgets.Dropdown(
    options=[("require exit", "require"), ("any", "any"), ("sealed", "forbid")],
    value="any",
    description="Opening:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="220px"),
)
new_btn = widgets.Button(description="New game", button_style="info")
end_btn = widgets.Button(description="End game")
preset = widgets.Dropdown(
    options=[
        ("Beginning, move", "begin_move"),
        ("Beginning, look", "begin_look"),
        ("Look at your target", "look_target"),
        ("Move to your target", "move_target"),
        ("Direction", "direction"),
        ("Hour", "hour"),
    ],
    value="begin_move",
    description="Preset:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="280px"),
)
question = widgets.Textarea(
    value=S2_BEGINNING_MOVE_USER,
    layout=widgets.Layout(width="100%", height="120px"),
)
turn_btn = widgets.Button(description="Agent turn", button_style="success", icon="play")
auto_btn = widgets.Button(description="Auto-play", button_style="warning")
replay_btn = widgets.Button(description="Replay", icon="refresh")
status = widgets.HTML("<span style='font-family:sans-serif'>Load a checkpoint.</span>")
readout = widgets.HTML("Click a token to draw s and v on the frame S2 saw.")
legend = widgets.HTML(
    "<span style='color:#1e5aff'>\u25cf s</span> &nbsp; "
    "<span style='color:#ff0000'>\u25cf v</span> &nbsp; "
    "<span style='color:#c0392b'>[MOVE] closes the reply</span> &nbsp; "
    "<span style='color:#1a7f37'>[END_GAME] ends the game</span>"
)
board = widgets.Image(format="png", width=BOARD_PX, height=BOARD_PX)
token_html = widgets.HTML()
conv = widgets.HTML("<div class='fs-msg'>No turns yet.</div>")
notepad = widgets.HTML("<pre>No game.</pre>")
log = widgets.Output(layout=widgets.Layout(
    border="1px solid #ccc", height="180px", overflow="auto",
))
pick_box = widgets.Text(value="")
pick_box.add_class("fs-pick")
pick_box.layout.height = "0px"
pick_box.layout.visibility = "hidden"
pick_box.layout.overflow = "hidden"
pick_box.observe(on_pick, names="value")
preset.observe(on_preset, names="value")

controls = [
    ckpt, refresh_btn, load_btn, gold, opening, new_btn, end_btn,
    preset, question, turn_btn, auto_btn, replay_btn,
]
refresh_btn.on_click(on_refresh)
load_btn.on_click(on_load)
new_btn.on_click(on_new)
end_btn.on_click(on_end)
turn_btn.on_click(on_turn)
auto_btn.on_click(on_auto)
replay_btn.on_click(on_replay)

_say(f"{len(_paths)} full checkpoint(s) under {weights_root() / 'full'}")
_show_tokens()

display(HTML(
    """
<style>
.fs-tokens {
  white-space: pre-wrap;
  line-height: 1.8;
  border: 1px solid #ccc;
  padding: 8px;
  max-height: 220px;
  overflow-y: auto;
  font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
}
.fs-tok { cursor: pointer; border-radius: 3px; }
.fs-tok:hover { background: #e8eefc; }
.fs-tok.fs-on { background: #ffe08a; }
.fs-eos { opacity: 0.55; }
.fs-move { box-shadow: inset 0 0 0 2px #c0392b; }
.fs-end { box-shadow: inset 0 0 0 2px #1a7f37; }
.fs-turn { font-family: sans-serif; font-size: 12px; color: #666; }
.fs-msg pre {
  white-space: pre-wrap;
  font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
  font-size: 13px;
}
.fs-log {
  max-height: 280px;
  overflow-y: auto;
  border: 1px solid #ccc;
  padding: 8px;
}
</style>
<script>
(function () {
  if (window.__fsTokenClick) { return; }
  window.__fsTokenClick = true;
  window.addEventListener("click", function (ev) {
    var span = ev.target && ev.target.closest && ev.target.closest(".fs-tok");
    if (!span) { return; }
    var box = document.querySelector(".fs-pick input");
    if (!box) { return; }
    window.__fsPickN = (window.__fsPickN || 0) + 1;
    var gen = span.getAttribute("data-gen") || "0";
    var turn = span.getAttribute("data-turn") || "0";
    var i = span.getAttribute("data-i");
    box.value = window.__fsPickN + "," + gen + "," + turn + "," + i;
    box.dispatchEvent(new Event("input", { bubbles: true }));
  }, true);
})();
</script>
"""
))
display(widgets.VBox([
    widgets.HBox([ckpt, refresh_btn, load_btn]),
    widgets.HBox([gold, opening, new_btn, end_btn]),
    preset,
    question,
    widgets.HBox([turn_btn, auto_btn, replay_btn, legend]),
    status,
    readout,
    widgets.HBox([
        board,
        widgets.VBox([
            widgets.HTML("<b>Notepad</b>"),
            notepad,
            widgets.HTML("<b>Conversation</b>"),
            widgets.VBox([conv], layout=widgets.Layout(
                height="280px", overflow="auto", border="1px solid #ccc",
            )),
        ], layout=widgets.Layout(flex="1 1 auto")),
    ], layout=widgets.Layout(align_items="flex-start")),
    token_html,
    log,
    pick_box,
]))
